# Introducción al análisis de datos censales con Python

## ¿De qué se trata esta notebook?

Esta guía es una introducción práctica al análisis de datos usando el **Censo Nacional de Población, Hogares y Viviendas 2022** de Argentina (INDEC).

No se necesita saber estadística avanzada ni tener experiencia previa con datos censales. Alcanza con conocer Python básico.

Al final de esta guía vas a poder:
- Consultar variables del censo directamente desde Python
- Obtener tablas con N y porcentajes
- Comparar indicadores entre provincias y departamentos
- Generar visualizaciones básicas

---

## ¿Qué es `censoargentino`?

Es un paquete Python que conecta directamente con los datos del censo procesados en formato Parquet y almacenados en [Hugging Face](https://huggingface.co/datasets/pedroorden/censoargentino). Usa DuckDB internamente para hacer consultas eficientes: **no descarga el dataset completo** (137 MB), solo los bloques que corresponden a tu consulta.

```
INDEC (base REDATAM)  →  procesamiento  →  Hugging Face  →  censoargentino  →  tu análisis
```

---
## 0. Instalación

Si todavía no tenés el paquete instalado:

In [ ]:
# Descomenta si necesitás instalar
# !pip install censoargentino

In [ ]:
import censoargentino as censo
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Estilo general de los gráficos
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print("Todo listo.")

---
## 1. ¿Qué datos tenemos?

Antes de analizar, conviene entender qué variables están disponibles. El censo 2022 tiene tres **entidades** principales:

| Entidad | Qué mide |
|---|---|
| `PERSONA` | Características de cada persona: sexo, edad, educación, actividad económica |
| `HOGAR` | Características del hogar: NBI, hacinamiento, acceso a servicios |
| `VIVIENDA` | Tipo y calidad de la vivienda, acceso a agua y saneamiento |

Usamos `censo.variables()` para ver el catálogo completo.

In [ ]:
# Ver todas las variables disponibles
vars_df = censo.variables()
print(f"Total de variables: {len(vars_df)}")
vars_df

In [ ]:
# Cuántas variables hay por entidad
vars_df.groupby("entidad").size().rename("variables").reset_index()

In [ ]:
# Buscar variables sobre educación
censo.variables(buscar="instruccion")

Cuando encontramos una variable que nos interesa, usamos `describe()` para entender exactamente qué mide y cuáles son sus categorías posibles.

In [ ]:
# ¿Qué mide exactamente PERSONA_MNI?
censo.describe("PERSONA_MNI")

---
## 2. Primera consulta: distribución de sexo

Empezamos con algo simple. La variable `PERSONA_P02` registra el sexo al nacer.

Usamos `censo.tabla()` que en un solo paso descarga los datos y calcula N y porcentaje.

In [ ]:
# Distribución de sexo a nivel nacional
sexo = censo.tabla("PERSONA_P02")
sexo

El resultado tiene tres columnas:
- **`categoria`**: el valor de la variable (en este caso, Mujer o Varón)
- **`N`**: cantidad de personas en esa categoría
- **`%`**: porcentaje sobre el total

Podemos visualizarlo en un gráfico de barras:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
colores = ["#e07b8a", "#5b8ec4"]
ax.bar(sexo["categoria"], sexo["N"] / 1e6, color=colores, alpha=0.85, width=0.4)

for i, (n, pct) in enumerate(zip(sexo["N"], sexo["%"])):
    ax.text(i, n / 1e6 + 0.1, f"{n/1e6:.1f}M  ({pct}%)", ha="center", fontsize=9)

ax.set_ylabel("Millones de personas")
ax.set_title("Distribución por sexo — Argentina (Censo 2022)")
ax.set_ylim(0, sexo["N"].max() / 1e6 * 1.2)
plt.tight_layout()
plt.show()

---
## 3. Filtrar por provincia

Todas las funciones aceptan un parámetro `provincia`. Podés pasar el nombre o el código INDEC de 2 dígitos.

In [ ]:
# Ver los códigos disponibles
censo.provincias()

In [ ]:
# Sexo en CABA (código "02") y en Tucumán (por nombre)
sexo_caba   = censo.tabla("PERSONA_P02", provincia="02")
sexo_tucuman = censo.tabla("PERSONA_P02", provincia="Tucumán")

print("=== CABA ===")
print(sexo_caba.to_string(index=False))
print("\n=== Tucumán ===")
print(sexo_tucuman.to_string(index=False))

---
## 4. Comparar provincias con `comparar()`

`comparar()` devuelve una tabla pivot donde cada fila es una provincia y cada columna es una categoría de la variable. Los valores son porcentajes.

Es ideal para ver patrones geográficos de un vistazo.

In [ ]:
# Porcentaje de hogares con NBI por provincia
# NBI = Necesidades Básicas Insatisfechas
nbi_prov = censo.comparar("HOGAR_NBI_TOT")
nbi_prov

La columna **`Total`** es la cantidad de hogares de esa provincia. Las demás columnas son el % en cada categoría.

Podemos graficar solo la columna que nos interesa:

In [ ]:
# Identificar la columna con NBI positivo (la que no dice "Sin")
col_nbi = [c for c in nbi_prov.columns if "Sin" not in c and c != "Total"][0]
print(f"Columna NBI: '{col_nbi}'")

plot_df = nbi_prov[[col_nbi]].sort_values(col_nbi)
media = plot_df[col_nbi].mean()

fig, ax = plt.subplots(figsize=(8, 8))
colores = ["#e8965a" if v > media else "#69b3a2" for v in plot_df[col_nbi]]
ax.barh(plot_df.index, plot_df[col_nbi], color=colores, alpha=0.85)
ax.axvline(media, color="grey", linestyle="--", linewidth=0.8,
           label=f"Promedio: {media:.1f}%")

for prov, val in plot_df[col_nbi].items():
    ax.text(val + 0.2, list(plot_df.index).index(prov), f"{val}%",
            va="center", fontsize=8)

ax.set_xlabel("% hogares con NBI")
ax.set_title("Hogares con Necesidades Básicas Insatisfechas\npor provincia — Argentina (Censo 2022)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Bajar al nivel de departamento

Cuando queremos más granularidad dentro de una provincia, usamos `nivel="departamento"`.

In [ ]:
# NBI por departamento dentro de Salta
# Salta tiene alta heterogeneidad interna
nbi_salta = censo.comparar("HOGAR_NBI_TOT", nivel="departamento", provincia="Salta")
nbi_salta

In [ ]:
# Graficamos los departamentos de Salta
col_nbi = [c for c in nbi_salta.columns if "Sin" not in c and c != "Total"][0]
plot_dpto = nbi_salta[[col_nbi]].sort_values(col_nbi)

fig, ax = plt.subplots(figsize=(7, 7))
ax.barh(plot_dpto.index, plot_dpto[col_nbi], color="#e8965a", alpha=0.85)
for dpto, val in plot_dpto[col_nbi].items():
    ax.text(val + 0.3, list(plot_dpto.index).index(dpto), f"{val}%",
            va="center", fontsize=8)
ax.set_xlabel("% hogares con NBI")
ax.set_title("NBI por departamento — Salta (Censo 2022)")
plt.tight_layout()
plt.show()

---
## 6. Datos crudos y agregación manual

Para análisis más avanzados, `query()` devuelve el dataset en **formato largo por radio censal**: la unidad mínima geográfica del censo (~52.000 radios en todo el país).

Cada radio es un área pequeña de entre 200 y 300 viviendas.

`agregar()` permite resumir ese dataset sin tener que volver a descargarlo.

In [ ]:
# Descargamos datos de educación para Misiones
df_educ = censo.query(variables="PERSONA_MNI", provincia="Misiones")

print(f"Filas (radios × categorías): {len(df_educ):,}")
print(f"Columnas: {list(df_educ.columns)}")
df_educ.head(4)

In [ ]:
# Totales provinciales — mismos datos, pero resumidos
educ_total = censo.agregar(df_educ)
# Excluimos la categoría "ignorado" (código 99)
educ_total = educ_total[~educ_total["categoria"].str.contains("gnor", case=False)]
educ_total

In [ ]:
# El mismo df, ahora abierto por departamento — sin nueva descarga
educ_dpto = censo.agregar(df_educ, por="departamento")
educ_dpto[~educ_dpto["categoria"].str.contains("gnor", case=False)].head(20)

---
## 7. Mini análisis: ¿Cuáles son las provincias con más privación material?

El IPMH (Índice de Privación Material de los Hogares) combina dos dimensiones:
- **Privación patrimonial**: vivienda inadecuada
- **Privación de recursos corrientes**: ingresos insuficientes

Tiene cuatro categorías: sin privación, solo patrimonial, solo recursos corrientes, y convergente (ambas).

In [ ]:
# Ver qué mide exactamente
censo.describe("HOGAR_IPMH")

In [ ]:
# Distribución nacional
ipmh_nac = censo.tabla("HOGAR_IPMH")
ipmh_nac

In [ ]:
# Comparación por provincia
ipmh_prov = censo.comparar("HOGAR_IPMH")

# Columna de privación convergente (la más grave: ambas privaciones)
col_conv = [c for c in ipmh_prov.columns if "Convergente" in c or "convergente" in c]
if not col_conv:
    col_conv = [c for c in ipmh_prov.columns if c != "Total"]
    print("Categorías disponibles:", list(ipmh_prov.columns))
else:
    col_conv = col_conv[0]
    print(f"Columna privación convergente: '{col_conv}'")
    ipmh_prov[[col_conv, "Total"]].sort_values(col_conv, ascending=False).head(10)

In [ ]:
# Gráfico comparativo — todas las categorías de IPMH por provincia
cols_plot = [c for c in ipmh_prov.columns if c != "Total"]
plot_data = ipmh_prov[cols_plot].sort_values(cols_plot[0], ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colores = ["#69b3a2", "#f0c27f", "#e8965a", "#c0392b"]
plot_data.plot(kind="barh", stacked=True, ax=ax,
               color=colores[:len(cols_plot)], alpha=0.85, width=0.7)

ax.set_xlabel("% hogares")
ax.set_title("IPMH por provincia — Argentina (Censo 2022)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

---
## 8. Para seguir explorando

### Algunas variables interesantes para investigar

```python
# Acceso a servicios básicos
censo.tabla("VIVIENDA_URP")               # urbano vs rural
censo.comparar("VIVIENDA_TIPOVIVG")       # tipos de vivienda por provincia

# Demografía
censo.tabla("PERSONA_EDADGRU")            # grandes grupos de edad
censo.comparar("PERSONA_CONDACT")         # actividad económica

# Déficit habitacional
censo.comparar("HOGAR_NBI_VIV")           # NBI vivienda
censo.comparar("HOGAR_NBI_SAN")           # NBI saneamiento
censo.tabla("HOGAR_H20CP", provincia="Formosa")  # hacinamiento en Formosa
```

### Recursos

- [Dataset en Hugging Face](https://huggingface.co/datasets/pedroorden/censoargentino)
- [Definiciones de variables — INDEC (PDF)](https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf)
- [Portal REDATAM online](https://redatam.indec.gob.ar/binarg/RpWebEngine.exe/Portal?BASE=CPV2022&lang=ESP)